In [ ]:
import ee, subprocess, geemap, time, json
import numpy as np
ee.Initialize()

## Step 1: Initialize Variables and Folders

In [ ]:
# Manually select a project ID that isn't already too full
project_id = 'your-project-id'
countryName = 'Rwanda'
countryNameString = countryName.replace(' ', '')
folder_name = f'{countryNameString.lower()}-candidate-locs'
path = f'projects/{project_id}/assets/{folder_name}'

In [ ]:
# Set up a folder structure:
ee.data.createAsset({'type': 'Folder'}, f'{path}')
time.sleep(0.2)
ee.data.createAsset({'type': 'Folder'}, f'{path}/S2')
time.sleep(0.2)
ee.data.createAsset({'type': 'Folder'}, f'{path}/S2/diffImgs')
ee.data.createAsset({'type': 'Folder'}, f'{path}/S2/locs')

## Step 2: Create grid cells, import roads, and construct masks

In [ ]:
# Use the Global Administrative Units dataset to get a geometry variable for our country
# Read more about the dataset here: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level1

countryShape = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(ee.Filter.eq('ADM0_NAME',countryName)) # get just the country in question from the GAUL dataset
countryGeo = countryShape.first().geometry().simplify(100) # get the geometry and simplify to reduce its size in memory

# Function to determine if an EE feature intersects a geometry
def intersectFeature(feature):
    return feature.intersection(countryGeo, 50)

# Create a covering grid of 50km squared for the country; eliminate portions that don't intersect with the country
grid = countryGeo.coveringGrid(countryGeo.projection(), 50000).map(intersectFeature)

# For the purposes of demonstration, take just the first few grid cells
grid = grid.limit(5)

In [ ]:
# Display the grid on a map (uncomment to use)
Map = geemap.Map(center=[-1.9403, 29.8739], zoom=7)
Map.addLayer(countryGeo, {}, 'Country Geometry')
Map.addLayer(grid, {'color': 'red'}, 'Grid Cells')
Map

## Step 3: Create a Mask of Populated Areas + Roads

In [ ]:
# Part 1: built-up areas

# We use the Global settlement characteristics dataset to get an idea of where human settlements exist
# Read more here: https://developers.google.com/earth-engine/datasets/catalog/JRC_GHSL_P2023A_GHS_BUILT_C

# Import the data, and create a mask based on a certain level of human infrastructure
builtup_characteristics_image = ee.Image("JRC/GHSL/P2023A/GHS_BUILT_C/2018").clip(countryGeo)
builtupThreshold = 4
mask = builtup_characteristics_image.gt(builtupThreshold)
mask = mask.updateMask(mask)

# Buffer the mask slightly, to enlarge it
bufferedMask = mask.focal_max(
  radius = 2,
  kernelType = 'octagon',
  iterations = 1
)

In [ ]:
# Convert the mask into a feature collection, since that is easier to work with
maskShape = bufferedMask.select('built_characteristics').reduceToVectors(
  scale = 50, #original: 100
  geometry = countryGeo,
  geometryType = 'polygon',
  eightConnected = False,
  labelProperty = 'built_characteristics',
  reducer = ee.Reducer.countEvery(),
  maxPixels = 1e13
).filterBounds(grid)

Map = geemap.Map()
Map.addLayer(maskShape, {'color': 'blue'}, 'Built-up Areas')
Map.centerObject(maskShape, 8)
Map

In [ ]:
# Part 2: roads

# Road data from https://download.geofabrik.de/
# Step 1: Download the zip file of all shapes 
# Step 2: Go to Earth Engine, use the upload asset tool to upload the shape file for "gis_osm_roads"

# Example of production code:
#roads = ee.FeatureCollection(f"my-asset-path/gis_osm_roads")
#roads = roads.filter(ee.Filter.inList('fclass', ['tertiary', 'secondary', 'primary', 'trunk'])).filterBounds(grid)

# For the purposes of this example, we will use a sample shape file
with open("rwanda_roads_sample.geojson", "r") as file:
    json_data = json.load(file)

roads = geemap.geojson_to_ee("rwanda_roads_sample.geojson")

Map = geemap.Map()
Map.addLayer(roads, {'color': 'blue'}, 'Roads')
Map.centerObject(roads, zoom=8)
Map

In [ ]:
# Merge the built-up areas and (buffered) roads into a single feature collection
def bufferFeature300(f): # orgiginal
    return f.buffer(300, 25).simplify(25).intersection(countryGeo, 25)

roadsBuffered = roads.map(bufferFeature300) # Buffer the road geometry by 300 meters, so we get the surrounding areas
maskShape = maskShape.merge(roadsBuffered)

In [ ]:
# Visualize the grid on a map to ensure everything looks correct
Map = geemap.Map()
Map.centerObject(grid)
Map.addLayer(maskShape, {'color': 'blue'})
Map

In [ ]:
# Export the built up area + roads mask to a feature collection in GEE
task = ee.batch.Export.table.toAsset(
  collection = maskShape,
  description = f'builtupMask_{countryNameString}',
  assetId = f'{path}/builtupMask'
);
task.start()

In [ ]:
# Export our grid to GEE
task = ee.batch.Export.table.toAsset(
  collection = grid,
  description = f'cells_{countryNameString}',
  assetId = f'{path}/cells_{countryNameString}'
);
task.start()

## Step 4: Run the processing steps in order

In [ ]:
# Establish node and ee-runner paths -- used to run our earth engine scripts
nodePath = str(subprocess.check_output(['which node'], shell=True))[2:-3]
eerunnerPath = str(subprocess.check_output(['which ee-runner'], shell=True))[2:-3]

In [ ]:
#a function to initiate an earth engine analysis script for one grid cell
def runProcess(countryName, cellNum, process):
    codeFile=f"./temp/{process}_{countryNameString}_{cellNum}.js"
    with open(f"./{process}.js", "r") as fin:
        with open(codeFile, "w") as fout:
            for line in fin:
                fout.write(
                    line.replace('INSERT_CELL_NUMBER_HERE', str(cellNum))
                   .replace('INSERT_COUNTRYNAME_HERE', countryName)
                    .replace('INSERT_PATH_HERE', path)
                )

    #run the GEE code
    subprocess.call([nodePath, eerunnerPath, codeFile, f"--project={project_id}"])

In [ ]:
# Step 1: Run 01_composite for each grid cell, and wait for all of these to finish

for cellNum in np.arange(grid.size().getInfo()):
    runProcess(countryName, cellNum, '01_composite')

In [ ]:
# Step 2: run 02_prep for each grid cell, and wait for all of these to finish

for cellNum in np.arange(grid.size().getInfo()):
    runProcess(countryName, cellNum, '02_prep')

In [ ]:
# Step 3: run 03_locGen for each grid cell, and wait for all of these to finish

for cellNum in np.arange(grid.size().getInfo()):
    runProcess(countryName, cellNum, '03_locGen')